# Python — PySpark for Data Engineering
---

<div style="padding:15px;border-left:8px solid #a18cd1;background:#f5f0ff;border-radius:4px;">
<strong>Core Insight:</strong> PySpark is pandas for data that doesn't fit in memory.
The critical lesson: Python UDFs are the bottleneck — they serialize data row-by-row
from JVM to Python (10x slower than built-in F. functions).
Replace Python UDFs with F.when().otherwise() chains.
</div>

### The Citi Migration
Citi's ETL pipeline migrated from pandas to PySpark when data volume exceeded 50GB/day.
Python UDFs were the bottleneck — serializing row-by-row. Replacing with `F.when().otherwise()`
cut runtime from 45 minutes to 4 minutes for the daily capacity aggregation.

## 🧠 Mental Models

| Model | The Insight |
|-------|-----------|
| **Lazy Evaluation** | PySpark builds a DAG of transformations. Nothing executes until you call an action (`.count()`, `.write()`). This lets Spark optimize the entire pipeline before running. |
| **The JVM Bridge** | Python UDFs cross the JVM→Python boundary row-by-row — like having 6M individual round-trips. Built-in F. functions run inside the JVM — no serialization. |
| **Immutable DataFrames** | Every transformation returns a NEW DataFrame. The original is unchanged. This enables Spark's optimization and parallelism. |

### Transformations vs Actions
```
Transformations (lazy — build the DAG):
  .filter(), .select(), .groupBy(), .join(), .withColumn()

Actions (trigger execution):
  .count(), .collect(), .show(), .write.parquet()
```

**Never call `.collect()` on a large DataFrame** — it brings ALL data to the driver.
Use `.write` to save results, or `.show(20)` to sample.

In [ ]:
# PySpark fundamentals — SparkSession, DataFrame creation
# Note: In Jupyter, SparkSession may already be available as 'spark'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType
from datetime import date

# Create SparkSession (one per application)
spark = SparkSession.builder     .appName("capacity-etl")     .config("spark.sql.shuffle.partitions", "200")     .getOrCreate()

# Schema definition — always define schema explicitly for production pipelines
monitoring_schema = StructType([
    StructField("server_id",    StringType(),  nullable=False),
    StructField("collected_at", DateType(),    nullable=False),
    StructField("metric_type",  StringType(),  nullable=True),
    StructField("metric_value", DoubleType(),  nullable=True),
    StructField("environment",  StringType(),  nullable=True),
])

# Create DataFrame from data (in prod: read from S3/Hive)
data = [
    ("SRV-001", date(2024, 1, 15), "cpu_pct",    72.5, "prod"),
    ("SRV-001", date(2024, 1, 15), "memory_pct", 55.2, "prod"),
    ("SRV-002", date(2024, 1, 15), "cpu_pct",    91.3, "prod"),
    ("SRV-002", date(2024, 1, 15), "memory_pct", 88.1, "prod"),
    ("SRV-003", date(2024, 1, 15), "cpu_pct",    23.0, "dev"),
]

df = spark.createDataFrame(data, schema=monitoring_schema)
print("Schema:")
df.printSchema()
print("\nRow count:", df.count())  # action — triggers execution
df.show()

In [ ]:
from pyspark.sql import functions as F

# ══════════════════════════════════════
# CORE OPERATIONS: filter, select, withColumn, groupBy
# ══════════════════════════════════════

# Read monitoring data (in prod: from S3 Parquet)
# df = spark.read.parquet("s3://citi-data/monitoring/date=2024-01-15/")

# ── Transformations (lazy) ────────────────────────────────────────────────────

# Filter to production CPU readings
prod_cpu = df.filter(
    (F.col("environment") == "prod") &
    (F.col("metric_type") == "cpu_pct")
)

# Add alert column using F.when (NOT a Python UDF)
with_alerts = prod_cpu.withColumn(
    "alert_level",
    F.when(F.col("metric_value") >= 90, "CRITICAL")
     .when(F.col("metric_value") >= 75, "WARNING")
     .when(F.col("metric_value") >= 60, "MONITOR")
     .otherwise("OK")
)

# Aggregations
summary = with_alerts.groupBy("server_id", "alert_level")     .agg(
        F.avg("metric_value").alias("avg_cpu"),
        F.max("metric_value").alias("peak_cpu"),
        F.count("*").alias("reading_count")
    )     .orderBy(F.col("avg_cpu").desc())

# ── Action: show results ───────────────────────────────────────────────────────
summary.show()

# ── Action: write to Parquet (production output) ───────────────────────────────
# summary.write #     .mode("overwrite") #     .partitionBy("alert_level") #     .parquet("s3://citi-data/capacity-summary/date=2024-01-15/")

In [ ]:
from pyspark.sql import functions as F

# ══════════════════════════════════════
# ❌ ANTI-PATTERN vs ✅ PATTERN: Python UDFs
# ══════════════════════════════════════

# ❌ ANTI-PATTERN: Python UDF — crosses JVM boundary for EVERY row
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(returnType=StringType())
def classify_cpu_python_udf(cpu_value):
    """This serializes EVERY row to Python and back — 10x slower."""
    if cpu_value is None:
        return "UNKNOWN"
    if cpu_value >= 90:
        return "CRITICAL"
    elif cpu_value >= 75:
        return "WARNING"
    elif cpu_value >= 60:
        return "MONITOR"
    return "OK"

# Slow: 45 minutes for 6M rows (JVM→Python→JVM for each row)
# df.withColumn("alert", classify_cpu_python_udf(F.col("metric_value")))

# ✅ PATTERN: Built-in F.when — stays inside the JVM, runs in C++
fast_classify = df.withColumn(
    "alert",
    F.when(F.col("metric_value").isNull(), "UNKNOWN")
     .when(F.col("metric_value") >= 90, "CRITICAL")
     .when(F.col("metric_value") >= 75, "WARNING")
     .when(F.col("metric_value") >= 60, "MONITOR")
     .otherwise("OK")
)

# Fast: 4 minutes for 6M rows (no serialization overhead)
fast_classify.show()

print("Key Rule: If you can express logic with F.when/F.col/F.expr, NEVER use a Python UDF")
print("Use Python UDFs ONLY for logic that has no equivalent built-in function")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ══════════════════════════════════════
# WINDOW FUNCTIONS in PySpark
# (Same concepts as SQL window functions)
# ══════════════════════════════════════

# Define window: for each server, ordered by date
server_window = Window.partitionBy("server_id").orderBy("collected_at")
server_30day_window = Window.partitionBy("server_id")     .orderBy("collected_at")     .rowsBetween(-29, 0)  # last 30 rows

# Add window function columns
df_with_windows = df.filter(F.col("metric_type") == "cpu_pct")     .withColumn("prev_day_cpu",
        F.lag("metric_value", 1).over(server_window))     .withColumn("rolling_30day_avg",
        F.avg("metric_value").over(server_30day_window))     .withColumn("deviation_from_baseline",
        F.col("metric_value") - F.col("rolling_30day_avg"))     .withColumn("rank_in_fleet",
        F.rank().over(Window.orderBy(F.col("metric_value").desc())))

df_with_windows.select(
    "server_id", "collected_at", "metric_value",
    "rolling_30day_avg", "deviation_from_baseline", "rank_in_fleet"
).show()

# ── Joins ───────────────────────────────────────────────────────────────────
# Broadcast join: when one table is small (< 200MB), broadcast to all executors
dim_server_df = spark.createDataFrame(
    [("SRV-001", "Team-Alpha"), ("SRV-002", "Team-Beta"), ("SRV-003", "Team-Dev")],
    ["server_id", "team"]
)

# Broadcast hint avoids shuffle for the small table
joined = df.join(F.broadcast(dim_server_df), on="server_id", how="left")
joined.show()

## 🎤 5 Interview Q&A

**Q1: What is the difference between a transformation and an action in PySpark?**
Transformations are lazy — they build a DAG but don't execute: `.filter()`, `.select()`, `.groupBy()`, `.join()`.
Actions trigger execution and return results: `.count()`, `.collect()`, `.show()`, `.write()`.
Spark's optimizer (Catalyst) sees the entire DAG before execution, enabling optimizations like
predicate pushdown (filter before join), projection pruning (only read needed columns), and
broadcast join selection.

---

**Q2: Why are Python UDFs slow in PySpark?**
PySpark runs on the JVM. Python UDFs require serializing each row from JVM memory → Python process,
executing the function, then deserializing the result back to JVM — for EVERY row.
On 6M rows, this is 6M serialization round-trips. Built-in F. functions (F.when, F.col, F.expr)
run natively in the JVM as optimized C++/Scala — no serialization. The speedup is typically 5-20x.
Use Python UDFs only when no built-in equivalent exists.

---

**Q3: What is a broadcast join and when should you use it?**
A broadcast join sends the entire small table to every executor, avoiding a shuffle.
Without a broadcast join, both tables are shuffled by the join key — expensive for large tables.
With `F.broadcast(small_df)`, the small table fits in memory on each executor and the large table
is processed locally. Use when one side is < 200MB (configurable via `spark.sql.autoBroadcastJoinThreshold`).
At Citi: dim_server (6,000 rows) vs fact_monitoring (500M rows) — always broadcast dim_server.

---

**Q4: What is the difference between `.cache()` and `.persist()`?**
Both store a DataFrame in memory for reuse. `.cache()` = `.persist(StorageLevel.MEMORY_AND_DISK)`.
`.persist()` lets you specify the storage level: MEMORY_ONLY (fastest), DISK_ONLY (slowest, largest),
MEMORY_AND_DISK (default — spills to disk if memory is full).
Use `.cache()` when you'll reuse a DataFrame multiple times in the same job (e.g., used in two
different branches of a pipeline). Unpersist when done: `df.unpersist()`.

---

**Q5: What is a shuffle and how do you minimize it?**
A shuffle redistributes data across the cluster so that records with the same key end up on the
same executor — required for `groupBy`, `join`, and `orderBy`. Shuffles involve writing data to
disk and transferring over the network — the most expensive Spark operation.
Minimize with: (1) filter early to reduce data before shuffle, (2) use broadcast joins for small tables,
(3) set `spark.sql.shuffle.partitions` to match data size (default 200 is often wrong), (4) avoid
unnecessary `orderBy` — use `sortWithinPartitions` if only local sort is needed.

## 📚 Key Terms

| Term | Definition |
|------|------------|
| **RDD** | Resilient Distributed Dataset — original Spark abstraction. Low-level, avoid in favor of DataFrame API |
| **DataFrame** | Distributed table with named columns and schema — the standard PySpark API |
| **Transformation** | Lazy operation that returns a new DataFrame — builds the DAG |
| **Action** | Triggers execution and returns results — count, collect, show, write |
| **DAG** | Directed Acyclic Graph — Spark's execution plan, optimized before running |
| **Catalyst Optimizer** | Spark's query optimizer — rewrites the DAG for efficiency (predicate pushdown, etc.) |
| **Shuffle** | Redistributing data across executors by key — expensive network + disk operation |
| **Broadcast Join** | Send small table to all executors to avoid shuffle — use when one side < 200MB |
| **Partition** | A chunk of data processed by one executor — default 200 after shuffle |
| **Python UDF** | User-defined function in Python — crosses JVM boundary, 5-20x slower than F. functions |
| **Predicate Pushdown** | Applying filters as early as possible (even in the file reader) — reduces data scanned |
| **Lazy Evaluation** | Transformations are not executed until an action is called — enables optimization |

## 💼 The Citi Narrative

**Context:** Daily capacity aggregation ETL — processes monitoring data for 6,000 servers,
calculates rolling averages, flags at-risk servers. Data volume: 50GB+ per day.

**The Problem:** Pipeline was taking 45 minutes, blocking the capacity team's 07:00 standup.
Performance profiling (Spark UI) showed 90% of time in one stage: the alert classification step.
Root cause: Python UDF for alert level classification.

**The UDF Problem:**
```python
@udf(returnType=StringType())
def classify(cpu):
    if cpu >= 90: return "CRITICAL"
    ...
```
6M rows × 1 serialization round-trip each = the bottleneck.

**The Fix — 2 Lines Changed:**
```python
# Before: Python UDF (45 min)
df.withColumn("alert", classify(F.col("metric_value")))

# After: F.when chain (4 min)
df.withColumn("alert", F.when(F.col("metric_value") >= 90, "CRITICAL").when(...).otherwise("OK"))
```

**Result:** 45 minutes → 4 minutes. No algorithm change. No infrastructure change.
The Spark UI showed the shuffle stage dropped from 40 minutes to 3 minutes.

**Interview Line:** *"The Spark UI showed one red stage taking 40 minutes. It was the UDF.
The function was in Python — Spark was serializing 6M rows from JVM to Python and back.
Replacing it with F.when took 5 minutes to write. The pipeline went from 45 to 4 minutes.
The lesson: check the Spark UI before optimizing anything."*

In [ ]:
# Practice: Rewrite these Python UDFs using F. functions

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, DoubleType

# ── PRACTICE 1: Categorize utilization ────────────────────────────────────────
# ❌ Python UDF (slow):
@udf(returnType=StringType())
def categorize_udf(value):
    if value is None: return None
    if value >= 90: return "CRITICAL"
    if value >= 70: return "HIGH"
    if value >= 50: return "MEDIUM"
    return "LOW"

# ✅ Write the F.when equivalent:
categorize_fwhen = None  # your answer

# ── PRACTICE 2: Apply a 10% headroom adjustment ────────────────────────────────
# ❌ Python UDF (slow):
@udf(returnType=DoubleType())
def add_headroom_udf(value):
    if value is None: return None
    return round(value * 1.10, 2)

# ✅ Write the F.col equivalent (one expression):
add_headroom_fwhen = None  # your answer

# ── Solutions ───────────────────────────────────────────────────────────────
categorize_fwhen = (
    F.when(F.col("metric_value").isNull(), None)
     .when(F.col("metric_value") >= 90, "CRITICAL")
     .when(F.col("metric_value") >= 70, "HIGH")
     .when(F.col("metric_value") >= 50, "MEDIUM")
     .otherwise("LOW")
)

add_headroom_fwhen = F.round(F.col("metric_value") * 1.10, 2)

print("Practice 1 solution: F.when chain (runs in JVM, no serialization)")
print("Practice 2 solution: F.round(F.col(...) * 1.10, 2)")
print("Key: ANY time you write a UDF, ask: can F.when/F.expr/F.col do this?")

## 🎯 Summary

### The Core Rules
1. **Transformations are lazy** — nothing runs until an action
2. **Never use Python UDFs when F. functions exist** — 5-20x slower
3. **Broadcast small tables** in joins — avoids expensive shuffles
4. **Filter early** — reduce data before groupBy/join operations
5. **Check Spark UI** — before optimizing, find the red stage

### Common Operations Cheatsheet
| Task | PySpark |
|------|---------|
| Filter rows | `.filter(F.col("env") == "prod")` |
| Add column | `.withColumn("alert", F.when(...))` |
| Aggregate | `.groupBy("server").agg(F.avg("cpu"))` |
| Join | `.join(F.broadcast(small_df), on="id")` |
| Window function | `.withColumn("lag", F.lag("cpu").over(w))` |
| Write output | `.write.mode("overwrite").parquet("s3://...")` |

### Interview Confidence Checklist
- [ ] Can explain transformation vs action with examples
- [ ] Can explain why Python UDFs are slow and how to avoid them
- [ ] Can write F.when chain to replace a Python UDF
- [ ] Can explain broadcast join and when to use it
- [ ] Can name the Citi narrative: 45 min → 4 min by removing Python UDF

**"Simplicity and clarity is Gold."** — Sean's Study Mantra 🚀